In [33]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, PatternedSequenceGenerator, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 
import os
import zipfile

In [38]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [143]:
class autoencoder(nn.Module):
    def __init__(self, input_size, projection_size):
        super().__init__()
        self.linear_en = nn.Linear(input_size, projection_size)
        self.linear_de = nn.Linear(projection_size, input_size)

    def forward(self, x):
        out_en = self.linear_en(x)
        #out_en_gated = nn.functional.relu(out_en)
        out_de = self.linear_de(out_en)

        return out_de

In [144]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [145]:
total_samples = 1000000
working_memory = 1
#hidden_size = 50
vocab_size = 7
embedding_dim = 2
lr = 1e-3


model = autoencoder(vocab_size, embedding_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_random_sequence(total_samples, token_number=vocab_size)
l = 0
t= 0 

for token in data:
    X = nn.functional.one_hot(torch.tensor(ord(data[0])- ord('A')), num_classes=vocab_size).float() 

    optimizer.zero_grad()
    out = model(X)

    loss = criterion(out, X)     
    loss.backward()
    optimizer.step()

    l += loss
    t += 1

    if t%1000 == 0:
        print('Loss: ', l/t)



Loss:  tensor(0.6813, grad_fn=<DivBackward0>)
Loss:  tensor(0.3530, grad_fn=<DivBackward0>)
Loss:  tensor(0.2374, grad_fn=<DivBackward0>)
Loss:  tensor(0.1787, grad_fn=<DivBackward0>)
Loss:  tensor(0.1432, grad_fn=<DivBackward0>)
Loss:  tensor(0.1194, grad_fn=<DivBackward0>)
Loss:  tensor(0.1024, grad_fn=<DivBackward0>)
Loss:  tensor(0.0896, grad_fn=<DivBackward0>)
Loss:  tensor(0.0797, grad_fn=<DivBackward0>)
Loss:  tensor(0.0717, grad_fn=<DivBackward0>)
Loss:  tensor(0.0652, grad_fn=<DivBackward0>)
Loss:  tensor(0.0598, grad_fn=<DivBackward0>)
Loss:  tensor(0.0552, grad_fn=<DivBackward0>)
Loss:  tensor(0.0512, grad_fn=<DivBackward0>)
Loss:  tensor(0.0478, grad_fn=<DivBackward0>)
Loss:  tensor(0.0448, grad_fn=<DivBackward0>)
Loss:  tensor(0.0422, grad_fn=<DivBackward0>)
Loss:  tensor(0.0398, grad_fn=<DivBackward0>)
Loss:  tensor(0.0377, grad_fn=<DivBackward0>)
Loss:  tensor(0.0359, grad_fn=<DivBackward0>)
Loss:  tensor(0.0342, grad_fn=<DivBackward0>)
Loss:  tensor(0.0326, grad_fn=<Div

In [146]:
with open('pretrained_w_en.pickle', 'wb') as f:
    pickle.dump(model, f)

In [151]:
class ShiftRegisterRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size, delay, file_to_load, activation='relu'):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.delay = delay
        self.W_in = nn.Linear(input_size, hidden_size)
        self.activation = getattr(F, activation, F.relu)  # use F.tanh, F.relu, etc.
        self.output_linear1 = nn.Linear(delay*hidden_size, 100)
        self.output_linear2 = nn.Linear(100, input_size)

        #with open(file_to_load, 'rb') as f:
        #    pretrained_model = pickle.load(f)

        #with torch.no_grad():
        #    for p2, p1 in zip(self.W_in.parameters(), pretrained_model.linear_en.parameters()):
        #        p2.copy_(p1)
        
        for param in self.W_in.parameters():
            param.requires_grad = False

    def forward(self, x, h_prev=None):
        """
        x: [batch, input_size]
        h_prev: [batch, delay, hidden_size]
        returns: h_new, output
        """
        B = x.size(0)
        device = x.device

        if h_prev is None:
            h_prev = torch.zeros(B, self.delay, self.hidden_size, device=device)

        # Compute new encoding for current input
        new_slot = self.W_in(x)  # [B, H]

        # Shift the register: move everything one step down
        h_new = torch.zeros_like(h_prev)
        h_new[:, 1:] = h_prev[:, :-1]  # shift
        h_new[:, 0] = new_slot         # insert newest on top

        # Output can be current state or readout (here we use last one)
        output = nn.functional.relu(self.output_linear1(h_new.view(-1)))
        output = self.output_linear2(output)

        return output, h_new



In [152]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, input_size):
        
        self.X = np.zeros((len(data)-1, input_size))
        self.y = np.zeros((len(data)-1))

        for ii in range(self.X.shape[0]-1):
            self.X[ii] = \
                nn.functional.one_hot(torch.tensor(ord(data[ii])- ord('A')), num_classes=input_size).float() 
      
            self.y[ii] = \
                ord(data[ii+1])- ord('A')

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [153]:
input_size = 7
hidden_size = 2
delay = 7
n_community = 2
n_member = 3
lr = 1e-3

rnn = ShiftRegisterRNNCell(input_size=input_size, hidden_size=hidden_size, delay=delay, file_to_load='pretrained_w_en.pickle')
optimizer = torch.optim.Adam(rnn.parameters(), lr=lr, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_sequence(total_samples, n_community, n_member, train=False)
data_set = Dataset_converter(data, input_size=input_size)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
test_acc = []
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        y_pred, h = rnn(X)
    else:
        y_pred, h = rnn(X, hidden)

    loss = criterion(y_pred.view(1,-1), y)     
    loss.backward()
    optimizer.step()

    # print(total)
    with torch.no_grad():
        hidden = h.detach()
        total += 1

        if y == y_pred.argmax():
            correct[total%1000] = 1
        else:
            correct[total%1000] = 0


        test_acc.append(
                np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        
        
        if total%10000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 10001, loss: 0.5556, accuracy: 0.8800
Iter : 20001, loss: 0.8359, accuracy: 0.8760
Iter : 30001, loss: 0.6636, accuracy: 0.8750
Iter : 40001, loss: 0.6506, accuracy: 0.8860
Iter : 50001, loss: 0.6904, accuracy: 0.8770
Iter : 60001, loss: 0.6789, accuracy: 0.8790
Iter : 70001, loss: 0.7653, accuracy: 0.8790
Iter : 80001, loss: 0.5841, accuracy: 0.8740


KeyboardInterrupt: 

In [126]:
h

tensor([[[-2.7068,  3.0743],
         [-2.7068,  3.0743],
         [-5.4137,  6.1487],
         [-2.7068,  3.0743],
         [-2.7068,  3.0743],
         [-2.7068,  3.0743],
         [-2.7068,  3.0743]]])

In [119]:
y_pred

tensor([-0.0712,  0.0815, -0.1180,  0.0413,  0.0371, -0.0561,  0.0827],
       grad_fn=<ViewBackward0>)

In [120]:
y

tensor([3])

In [17]:
rnn = ShiftRegisterRNNCell(input_size=4, hidden_size=2, delay=5)
h = None
for t in range(10):
    x = t*torch.ones(1, 4)  # batch of 2
    h, y = rnn(x, h)
    print(f"Step {t}, y.mean()={y.mean():.3f}")

Step 0, y.mean()=0.000
Step 1, y.mean()=0.000
Step 2, y.mean()=0.000
Step 3, y.mean()=0.000
Step 4, y.mean()=0.374
Step 5, y.mean()=0.310
Step 6, y.mean()=0.151
Step 7, y.mean()=0.055
Step 8, y.mean()=0.017
Step 9, y.mean()=0.005
